# CA6011 Deep Learning for NLP: Week 7 Lab -- Transformers (BERT Finetuning)


In this lab, we'll learn to finetune a pretrained BERT-Style (Encoder-only) transformer model for sequence classification task. we'll cover key functionalties used in Huggingface Transformers library to train/finetune Transformers models like [Trainer](https://huggingface.co/docs/transformers/main_classes/trainer#trainer), and [TrainingArguments](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments), which we will explain them in details in their corresponding sections below. For detailed information, refer to the Week 7 lecture slides and [Speech and Language Processing (3rd ed. draft) Dan Jurafsky and James H. Martin, Chapter 11](https://web.stanford.edu/~jurafsky/slp3/11.pdf).

This Lab uses the dataset used in [HuggingFace Token Classification Tutorial](https://huggingface.co/docs/transformers/v4.49.0/en/tasks/token_classification).


In this Lab, we also, show training models using both PyTorch and the Hugging Face API. With PyTorch, developers are required to manually code various components, offering complete control over the training process. In contrast, the Hugging Face API abstracts and optimises many underlying details, providing a more user-friendly interface for model training.

In [1]:
# Import the clear_output function from IPython.display module for clearing Jupyter Notebook cell output
from IPython.display import clear_output
# Install the necessary libraries for handling datasets and tokenization
# Tokenizers library from Hugging Face, specifically for creating and training custom tokenizers
!pip install transformers evaluate seqeval
!pip install -U "datasets<4.0.0"
!pip install 'sympy>=1.13.1,<1.14'
!pip install 'accelerate>=1.1.0'
clear_output()

In [2]:
import numpy as np

import torch
import torch.nn as nn
from torch.nn import CrossEntropyLoss
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import precision_recall_fscore_support
from sklearn.metrics import accuracy_score

from transformers import BertModel, BertTokenizerFast
from transformers import TrainingArguments
from transformers import Trainer

from transformers import DataCollatorForTokenClassification

import evaluate
from datasets import load_dataset

from tqdm import tqdm

import time

from typing import List, Optional, Tuple, Union


# Check if CUDA (GPU acceleration) is available on the system
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

# set a model name as in HuggingFace Model Hub, we will use to load the model weights.
# in this lab we will use a small size pretrained model (BERT-base)
pretrained_model_name = 'bert-base-uncased'

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
import os
os.environ["WANDB_DISABLED"]='true'

# 1 Extend Pretrained BERT with a Classifier

![](https://drive.google.com/uc?export=view&id=1AbA_sisJhjaeQ9QdSkHRsVRpCV6ByhWp)

In [4]:
class BertClassifier(nn.Module):
    def __init__(self, bert_model: BertModel, num_labels: int):
        super(BertClassifier, self).__init__()
        """
          Parameters:
            - bert_model (BertModel): Pretrained BERT model to initialise our classifier.
            - num_labels (int): The number of labels for our classifier's output.

          Attributes:
            - bert (BertModel): BERT model.
            - num_labels (int): Number of labels in output.
            - classifier (nn.Linear): Linear transformation for the output of the
                pretrained model to get the labels prediction.
            - dropout (nn.Dropout): Dropout layer to prevent overfitting.
        """

        # use the model given in input to the init function, which is a pretrained BERT model
        self.bert = bert_model

        # Number of labels in the sequence classification task
        self.num_labels = num_labels

        # We can access the information like hidden size, vocab, #layers, ect..
        # of a pretrained model in its config structure, which is accessible at model.config
        # https://huggingface.co/docs/transformers/model_doc/bert#transformers.BertConfig

        # Classifier layer to map the output of BERT to label space
        self.classifier = nn.Linear(bert_model.config.hidden_size, num_labels)

        # Dropout layer to prevent overfitting
        # here we used the same dropout value as the model hidden states dropout
        self.dropout = nn.Dropout(bert_model.config.hidden_dropout_prob)


    def forward(self, input_ids: torch.Tensor,
                attention_mask: torch.Tensor,
                token_type_ids: torch.Tensor,
                labels: torch.Tensor=None):
        """
        Forward pass for the BERT-based classifier.

        Parameters:
        - input_ids (torch.Tensor): tensor containing the ids of the tokenised input.
        - attention_mask (torch.Tensor): tensor containing the attention mask of the tokenised input.
        - token_type_ids (torch.Tensor): tensor containing the token type ids of the tokenised input.
        - labels (torch.Tensor): tensor containing the aligned target labels.

        Returns:
        - torch.Tensor: Output of the classifier.
        """

        # Get the outputs from pretrained BERT
        outputs = self.bert(input_ids=input_ids,
                            attention_mask=attention_mask,
                            token_type_ids=token_type_ids)

        # we will use the last hidden state information, since it's an accumulation of processed information in previous layers
        # Some practitioner, instead of the last hidden state, they take all layers hidden states and average them.
        # For many tasks last hidden state works the best.
        # BERT model also return all hidden states and you can access it using outputs.hidden_states
        sequence_output = outputs.last_hidden_state
        sequence_output = self.dropout(sequence_output)
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        output = (logits,) + outputs[2:]
        return ((loss,) + output) if loss is not None else output


# 2 Prepare Finetuning Dataset


For more information on how to prepare a dataset check this [YouTube tutorial](https://youtu.be/iY2AZYdZAr0).

To fine-tune our BERT model, we use [wnut_17 dataset](https://huggingface.co/datasets/wnut_17).

This dataset focuses on the task of "Emerging and Rare Entity Recognition." It is particularly aimed at identifying unusual, previously unseen entities within the context of emerging discussions. The recall of these entities can be challenging, especially when they are rare or emerging in nature.

It is composed of *tokens* and respective *NER tags*, which are expressed as int values that describe entities, such as corporation, location, or person.

Each tag value is associated to a specific tag:

- 0: O
- 1: B-corporation
- 2: I-corporation
- 3: B-creative-work
- 4: I-creative-work
- 5: B-group
- 6: I-group
- 7: B-location
- 8: I-location
- 9: B-person
- 10: I-person
- 11: B-product
- 12: I-product

NER tags are represented in **IOB format** in which the letter that prefixes each tag indicates the token position of the entity:

- B: beginning of the entity;
- I: (Inside) a token is contained inside the same entity;
- O: the token doesn't correspond to any entity.

The dataset is divided in train, validation and test set as follows:


Split | Size
------|-----
Train | 3394
Validation  | 1009
Test  | 1287

In [5]:
wnut = load_dataset("wnut_17", trust_remote_code=True)  # Load the dataset from HuggingFace

In [6]:
# Checking the first element in the dataset, we can see that each token has an associated NER tag expressed as int values
wnut["train"][0]

{'id': '0',
 'tokens': ['@paulwalk',
  'It',
  "'s",
  'the',
  'view',
  'from',
  'where',
  'I',
  "'m",
  'living',
  'for',
  'two',
  'weeks',
  '.',
  'Empire',
  'State',
  'Building',
  '=',
  'ESB',
  '.',
  'Pretty',
  'bad',
  'storm',
  'here',
  'last',
  'evening',
  '.'],
 'ner_tags': [0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  7,
  8,
  8,
  0,
  7,
  0,
  0,
  0,
  0,
  0,
  0,
  0,
  0]}

In [7]:
# Print the list of NER tags available in the dataset
# The position of each tag gives us the int values of NER tags in the dataset.
label_list = wnut["train"].features[f"ner_tags"].feature.names
print('Available NER tags:', label_list)

Available NER tags: ['O', 'B-corporation', 'I-corporation', 'B-creative-work', 'I-creative-work', 'B-group', 'I-group', 'B-location', 'I-location', 'B-person', 'I-person', 'B-product', 'I-product']


In [8]:
# The number of available NER tags is the number of labels needed in our classifier,
# i.e. the number of labels that our classifier will need to predict
num_labels = len(label_list)
print('Number of NER tags in the dataset:', num_labels)

Number of NER tags in the dataset: 13


To allow our model to process the dataset, we need to transform our input text into features that our network can understand.

In order to do this, we use a tokenizer that takes in input our text and returns the input ids, the token type ids and the attention mask associated to the given text.

In [9]:
# Load the pretrained BERT tokenizer from HuggingFace
tokenizer = BertTokenizerFast.from_pretrained(pretrained_model_name)

In [10]:
# Let's tokenize the first sample in the train set to check what happens when it's tokenized
tokenized_input = tokenizer(wnut['train'][0]["tokens"], is_split_into_words=True)

# Convert the tokenized input from token ids to string tokens
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])

In [11]:
print('Tokenized input:', tokenized_input)

print('\nInput:', wnut['train'][0]["tokens"])
print('Tokenized input ids:', tokenized_input['input_ids'])
print('Tokenized input tokens:', tokens)

print('\nLength of Input:', len(wnut['train'][0]["tokens"]))
print('Length of Tokenized input:', len(tokenized_input['input_ids']))
print('Length of Target labels:', len(wnut['train'][0]["ner_tags"]))

Tokenized input: {'input_ids': [101, 1030, 2703, 17122, 2009, 1005, 1055, 1996, 3193, 2013, 2073, 1045, 1005, 1049, 2542, 2005, 2048, 3134, 1012, 3400, 2110, 2311, 1027, 9686, 2497, 1012, 3492, 2919, 4040, 2182, 2197, 3944, 1012, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

Input: ['@paulwalk', 'It', "'s", 'the', 'view', 'from', 'where', 'I', "'m", 'living', 'for', 'two', 'weeks', '.', 'Empire', 'State', 'Building', '=', 'ESB', '.', 'Pretty', 'bad', 'storm', 'here', 'last', 'evening', '.']
Tokenized input ids: [101, 1030, 2703, 17122, 2009, 1005, 1055, 1996, 3193, 2013, 2073, 1045, 1005, 1049, 2542, 2005, 2048, 3134, 1012, 3400, 2110, 2311, 1027, 9686, 2497, 1012, 3492, 2919, 4040, 2182, 2197, 3944, 1012, 102]
Tokenized input tokens: ['[CLS]', '@', 'paul', '##walk', 'it', "'", 's', '

As we can see, the length of our target labels and the tokenized input are different because BERT tokenizer might split input words into multiple subwords. However, the target labels correspond to entire words, not subwords. For this reason, it's crucial to align the target labels with the tokens produced by the tokenizer.


`tokenize_and_align_labels` function, iterates over each sample to align labels with the tokenized input.

`word_ids` function call retrieves the index of the original word each token belongs to, allowing for alignment between labels and tokens.
 - Tokens not associated with any word (like `[CLS]`, `[SEP]`, `[PAD]`) get a label of -100, indicating they should be ignored in loss computation.
 - For words split into multiple tokens, only the first token receives the original label. Subsequent tokens (subwords) are labeled with -100 to avoid double-counting the word's label.

In [12]:
def tokenize_and_align_labels(examples, tokenizer: BertTokenizerFast):
    """
    Tokenize the input samples using the given tokenizer and
    align the target labels with the tokens produced by the tokenizer.

    Parameters:
    - samples (): Input tensor of shape [batch_size, seq_len, embed_size].
    - tokenizer (BertTokenizerFast): tokenizer used to tokenize the data

    Returns:
    - DatasetDict: Output the dataset containing the aligned target labels.

    Hint:
    - Remember that BERT's tokenizer may split a single word into multiple subword tokens.
      This means the number of tokens != the number of original words, so labels must be re-aligned.
    - Use tokenized_inputs.word_ids(batch_index=i) to get, for each token, the index of the
      original word it came from. Special tokens ([CLS], [SEP], [PAD]) will return None.
    - Only assign the real label to the FIRST subword token of each word.
      All other subword tokens (and special tokens) should get label -100,
      which tells PyTorch to ignore them in the loss computation.
    """
    ## INSERT YOUR CODE HERE ##
    # Tokenize the tokens in the dataset
    tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

    labels = [] # list of new labels
    for i, label in enumerate(examples[f"ner_tags"]):
        # Map each token in the tokenized input to their respective word in the original sample words,
        # i.e. you obtain a list containing for each output token the index of the associated original word.
        # If the index is None, it means a special token has been added to the input.
        # Hint: word_ids() returns a list where each position corresponds to a token,
        # and the value is the index of the original word that token came from.
        word_ids = tokenized_inputs.word_ids(batch_index=i)   # Map tokens to their respective word.

        previous_word_idx = None  # index of the last checked word
        label_ids = []
        for word_idx in word_ids:
            # Set the special tokens to -100 to be ignored in loss computation.
            if word_idx is None:
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # This is the FIRST token of a new word.
                # Assign the real label for this word.
                label_ids.append(label[word_idx])
            else:
                # This is a continuation subword token (same word as previous token).
                # We do NOT want to double-count this word's label, so assign -100.
                label_ids.append(-100)
            # Updated the id of the last checked word
            previous_word_idx = word_idx
        labels.append(label_ids)

    # Add the aligned labels back into the tokenized_inputs dictionary in the 'labels' field
    tokenized_inputs["labels"] = labels
    ## END OF YOUR CODE ##
    return tokenized_inputs

In [13]:
# Map the dataset to tokens and aligned labels using the tokenize_and_align_labels function
# batched=True is used to speed up the map function by processing multiple elements of the dataset at once
# fn_kwargs is used to pass multiple parameters to the tokenize_and_align_labels function
tokenized_wnut = wnut.map(tokenize_and_align_labels, batched=True, fn_kwargs={'tokenizer': tokenizer})

In [14]:
# Dataset before mapping and target label alignment
wnut

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 1287
    })
})

In [15]:
# Dataset after mapping and alignment
tokenized_wnut

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3394
    })
    validation: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1009
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1287
    })
})

After the mapping, we can see that the mapped dataset contains more columns than the original dataset, as we added the tokenized input, the token type ids, the attention mask and the aligned labels for each sample in the dataset.

# 3 Training with Torch

## 3.1 Dataset Class

Our Dataset class `WNUTDataset` will recieve the processed dataset from Section 2 and wrap it on a Torch `Dataset` class with the important specifications (contructor: `__init__`, length: `__len__`, Iterator: `__getitem__`) as outlined in Pytorch Tutorials in previous Labs.

In [16]:
class WNUTDataset(Dataset):
    def __init__(self, dataset):
        self.dataset = dataset

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        # Extract relevant data for the model: 'input_ids', 'attention_mask', 'token_type_ids', 'labels'
        item = {
            'input_ids': torch.tensor(self.dataset['input_ids'][idx]),
            'attention_mask': torch.tensor(self.dataset['attention_mask'][idx]),
            'token_type_ids': torch.tensor(self.dataset['token_type_ids'][idx]),
            'labels': torch.tensor(self.dataset['labels'][idx])
          }
        return item

Now, we will create Torch Dataset objects for our train, validation and test sets using the `WNUTDataset` class

In [17]:
train_dataset_torch = WNUTDataset(tokenized_wnut["train"])
eval_dataset_torch = WNUTDataset(tokenized_wnut["validation"])
test_dataset_torch = WNUTDataset(tokenized_wnut["test"])

Now we need to create a custom data collator, to ensure different samples in a batche have the same length.

**Ok, what's a Data Collator in the first place?**

After constructing your Dataset class, the Data Collator allows you to dictate the manner in which your inputs are batched. In our previous Lab sessions, we relied on the default Data Collator implementation in PyTorch. However, many NLP tasks necessitate a specific Data Collator class to address challenges such as sequences of varying lengths. For instance, imagine you have 10 samples divided into pairs, with each pair having different length, you might end up batching samples of different lengths together, necessitating padding for the shorter sequences to align with the lengths of the others.

In [18]:
def torch_collate_fn(batch):
    """
    Custom collate function to dynamically pad the batch so all
    input sequences have the same length.

    Parameters:
     - batch: collection of samples

    returns:
     - Dict: A dict of collated batches
    """
    # Extract input_ids, attention_mask, token_type_ids, and labels from the batch
    input_ids = [item['input_ids'] for item in batch]
    attention_mask = [item['attention_mask'] for item in batch]
    token_type_ids = [item['token_type_ids'] for item in batch]
    labels = [item['labels'] for item in batch]

    # Pad sequences dynamically
    inputs = tokenizer.pad(
        {"input_ids": input_ids, "attention_mask": attention_mask, "token_type_ids": token_type_ids},
        return_tensors="pt"
    )

    #Now we need to set labels for the pad tokens we just added.
    #get the padded input length
    sequence_length = inputs["input_ids"].shape[1]

    # check which side we did the padding and pad to labels to that side accordingly
    padding_side = tokenizer.padding_side
    label_pad_token_id = -100 #we ignore pad tokens
    if padding_side == "right":
            labels = [
                list(label) + [label_pad_token_id] * (sequence_length - len(label)) for label in labels
            ]
    else:
            labels = [
                [label_pad_token_id] * (sequence_length - len(label)) + list(label) for label in labels
            ]

    # Convert labels to a tensor
    labels = torch.tensor(labels, dtype=torch.long)

    # Update inputs dict to include labels
    inputs.update({"labels": labels})

    return inputs

## 3.2 Initialise our Model

In [19]:
# Let's initialise our classification model
bert_model_torch = BertModel.from_pretrained(pretrained_model_name) # Load a pretrained BERT model
num_labels = len(wnut["train"].features[f"ner_tags"].feature.names)  # Update this based on your NER task

model_torch = BertClassifier(bert_model_torch, num_labels=num_labels).to(device)  # init our classifier model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8794.39it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 3.3 Set Hyperparameters & Optimisers (Training Arguments)

In [20]:
#Batch size
Train_BATCH_SIZE = 16 #Batch size for training
Eval_BATCH_SIZE = 64 #Batch size for validation

# Learning Rate
learning_rate = 5e-5

We will use Adam optimser, same as the Week 5, 6 Labs

In [21]:
#Optimiser
optimizer = optim.AdamW(model_torch.parameters(), lr=learning_rate)

## 3.4 Training Loop

Let's first define a train function

In [22]:
def train_iter(model, iterator, optimizer, clip):

    model.train()

    epoch_loss = 0

    dataloader = DataLoader(iterator,
                            batch_size=Train_BATCH_SIZE,
                            collate_fn=torch_collate_fn,
                            shuffle=True,
                            num_workers=0, # tune: 2,4,8,...
                            pin_memory=True, # helps CPU->GPU copy
                            )

    for batch in tqdm(dataloader, total=len(dataloader), desc='Training'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()

        outputs = model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, labels=labels)
        loss = outputs[0]
        logits = outputs[1]

        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), clip)

        optimizer.step()

        epoch_loss += loss.item()


    return epoch_loss / len(dataloader)

Now, we need an evaluate function, before start training

In [23]:
@torch.no_grad()
def eval_iter(model, iterator):

    model.eval()
    epoch_loss = 0

    dataloader = DataLoader(iterator, batch_size=Eval_BATCH_SIZE, collate_fn=torch_collate_fn)

    for batch in tqdm(dataloader, total=len(dataloader), desc='Evaluation'):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        token_type_ids = batch['token_type_ids'].to(device)
        labels = batch['labels'].to(device)


        outputs = model(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, labels=labels)

        loss = outputs[0]
        logits = outputs[1]

        epoch_loss += loss.item()

    return epoch_loss / len(dataloader)

Next, we'll create a function that we'll use to tell us how long an epoch takes.

In [24]:
def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [25]:
# N_EPOCHS = 3
# CLIP = 1

# best_valid_loss = float('inf')

# for epoch in range(N_EPOCHS):

#     start_time = time.time()

#     train_loss = train_iter(model_torch, train_dataset_torch, optimizer, CLIP)
#     valid_loss = eval_iter(model_torch, eval_dataset_torch)

#     end_time = time.time()

#     epoch_mins, epoch_secs = epoch_time(start_time, end_time)

#     if valid_loss < best_valid_loss:
#         best_valid_loss = valid_loss
#         torch.save(model_torch.state_dict(), 'BERT-Finetuned-model-WNUT.pt')

#     print(f'Epoch: {epoch+1:02} | Time: {epoch_mins}m {epoch_secs}s')
#     print(f"\tTrain Loss: {train_loss:.3f})")
#     print(f"\tVal. Loss: {valid_loss:.3f})")


# 4 Training with HuggingFace API

## 4.1 Dataset Class

HuggingFace will create a torch Dataset class automatically, we just need to pass the datasets to a class called `Trainer` (Section 4.4).


**Note:** We need to ensure that our datasets contains columns with names like `input_ids` and `labels` as HuggingFace codebase expects the dataset contains to these columns.



In [26]:
train_dataset_hf, eval_dataset_hf, test_dataset_hf = tokenized_wnut["train"], tokenized_wnut["validation"], tokenized_wnut["test"]

HuggingFace's APIs, similar to PyTorch, enable you to utilise a Data Collator class to customise the batching of your inputs according to your requirements.


HuggingFace has created many Data Collator classes for different tasks, here we will use `DataCollatorForTokenClassification` class, as it suited for our task.

The [DataCollatorForTokenClassification](https://huggingface.co/docs/transformers/v4.17.0/en/main_classes/data_collator#transformers.DataCollatorForTokenClassification) creates a batch of examples, dynamically padding the text and labels to the length of the longest element in the batch, so they are a uniform length. This is more efficient than pad the text in the tokenizer function.

In [27]:
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

## 4.2 Initialise our Model

In [28]:
# Let's initialise our classification model
bert_model = BertModel.from_pretrained(pretrained_model_name) # Load a pretrained BERT model
num_labels = len(wnut["train"].features[f"ner_tags"].feature.names)  # Update this based on your NER task

model = BertClassifier(bert_model, num_labels=num_labels).to(device)  # init our classifier model

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 22410.76it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## 4.3 Training Arguments

**TrainingArguments** provides a structured way to configure training hyperparameters, such as learning rate, batch size, number of epochs, logging behaviour, and much more.

For more details check [here](https://huggingface.co/docs/transformers/main_classes/trainer#transformers.TrainingArguments)

In [29]:
training_args = TrainingArguments(
    output_dir='./model_output',    # Directory where model checkpoints and outputs will be saved.
    num_train_epochs=3,# Total number of training epochs.
    per_device_train_batch_size=16, # Batch size per device during training.
    per_device_eval_batch_size=64,  # Batch size for evaluation.
    learning_rate=5e-5,             # Learning rate
    warmup_steps=500,               # Number of warmup steps for learning rate scheduler.
    weight_decay=0.01,              # Weight decay if we apply some.
    logging_dir='./logs',           # Directory for storing logs.
    logging_steps=10,               # Log every X updates steps.
    eval_strategy="steps",    # Evaluate every X steps.
    eval_steps=50,                  # Number of steps to evaluate after.
    save_strategy="steps",          # The checkpoint save strategy to use.
    save_steps=100,                 # Save checkpoint every X steps.
    load_best_model_at_end=True,     # Whether to load the best model found at each evaluation.
    report_to="none"                 # The list of integrations to report the results and logs to.
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 4.4 compute_metrics

`compute_metrics` is a function used with Hugging Face's Trainer class to evaluate the performance of a model on a given dataset using metrics such as accuracy, precision, recall, F1 score. It's only used during evaluation, mostly practitioners use it to save the best model during training, and to track the important metrics during the training process.



In [30]:
metric = evaluate.load("seqeval")

def compute_metrics(model_output):
    """
    Compute evaluation metrics to check the performance of the model during training (on the validation set)
    and after training (on the test set).
    The input parameter of the function is a Tuple as required by the HuggingFace Trainer.

    Parameters:
    - model_output (Tuple): Contains model's raw predictions and target labels.

    Returns:
    - dict: Dictionary of the evaluation metrics, i.e. Precision, Recall, F1, and Accuracy.

    Returns:
    - dict: Dictionary of the evaluation metrics, i.e. Precision, Recall, F1, and Accuracy.

    Hint:
    - model_output is a tuple of (predictions, labels). predictions contains raw logits of shape
      [num_samples, seq_len, num_labels],  one score per label per token position.
    - To get the predicted class for each token, you need to pick the index of the highest logit.
      Think about which axis to apply np.argmax on.
    - Both predictions and labels still contain positions for special tokens ([CLS], [SEP], [PAD]).
      These were assigned label -100 during preprocessing, and must be filtered out before
      computing metrics. seqeval expects string label names (e.g. "B-person"), not integers.
    - label_list is available in the notebook scope and maps integer indices to string label names.
    """
    ## INSERT YOUR CODE HERE ##
    # Unpack model_output into predictions (raw logits) and labels.
    predictions, labels = model_output

    # Convert logits to predicted class indices by selecting the index of the max logit in each token position.
    predictions = np.argmax(predictions, axis=2)

    # Filter out the special tokens and align the predictions with the actual labels.
    # This step is necessary because models like BERT use special tokens (e.g., [CLS], [SEP], [PAD]),
    # and labels for these tokens are typically set to an ignore index (e.g., -100) so they don't affect loss computation.
    true_predictions = [
    [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
    for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    # Compute evaluation metrics such as precision, recall, F1 score, and accuracy using the 'metric' object.
    # This 'metric' object is typically an instance from the Hugging Face's `datasets` library, which provides
    # various metrics calculation functions.
    # Compute the metrics using the preloaded seqeval metric object.
    results = metric.compute(predictions=true_predictions, references=true_labels)

    # Return a dictionary containing the computed metrics.
    return {
    "precision": results["overall_precision"],
    "recall": results["overall_recall"],
    "f1": results["overall_f1"],
    "accuracy": results["overall_accuracy"],
}
  ## END OF YOUR CODE ##


## 4.5 Trainer

The Trainer class in Hugging Face's Transformers library is designed to simplify the training, evaluation, and testing of transformer models by providing a simple API where you only need to provide your model, training and evaluating sets, training arguments and an optional compute metrics function.

For more details, check [here](https://huggingface.co/docs/transformers/main_classes/trainer#trainer)

In [31]:
trainer = Trainer(
    model=model,                         # The model to be trained or fine-tuned.
    args=training_args,                  # TrainingArguments object containing the training and evaluation configurations.
    train_dataset=train_dataset_hf,         # The dataset to be used during training. Should be a Hugging Face Dataset or a dataset that implements __len__ and __getitem__.
    eval_dataset=eval_dataset_hf,           # The dataset for evaluation. Similar format as train_dataset. Used to evaluate the model performance at each logging step or epoch end.
    compute_metrics=compute_metrics,     # A function that computes metrics of interest for evaluation. It takes an EvalPrediction object (which has .predictions and .label_ids attributes) and should return a dictionary mapping metric names to their values.
    data_collator=data_collator          # Ensures samples are properly padded and batched
    )


## 4.6 Calling train()

In [ ]:
# After initializing our training pipeline with Trainer, we can start training
# by simply using the train function
# train() will train the specified model using the train_dataset given in input in the initialisation,
# evaluate the trained network every x number of steps using the validation set (eval_dataset)
# and save/load the best model at the end of the training
train_output = trainer.train()

/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Step,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
50,1.156483,1.098830,0.000000,0.000000,0.000000,0.920529
100,0.300576,0.394409,0.000000,0.000000,0.000000,0.920529
150,0.178676,0.286035,0.495305,0.252392,0.334390,0.936360
200,0.175747,0.250328,0.555773,0.339713,0.421678,0.940810
250,0.150462,0.266582,0.660156,0.404306,0.501484,0.947486
300,0.107056,0.225989,0.677007,0.443780,0.536127,0.949075
350,0.139873,0.198564,0.588633,0.520335,0.552381,0.951809
400,0.091820,0.223430,0.684685,0.454545,0.546370,0.952190
450,0.083198,0.226661,0.653846,0.569378,0.608696,0.957976
500,0.058532,0.216493,0.595579,0.547847,0.570717,0.954543


/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
/Users/robertsparks/Documents/neural_langu

## 4.7 Calling evaluate()

In [33]:
# evaluate the best trained model on the validation set
print('Evaluation metrics on the validation set:')
trainer.evaluate()

Evaluation metrics on the validation set:


/Users/robertsparks/Documents/neural_language_modelling/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


RuntimeError: on_train_begin must be called before on_evaluate

## 4.8 Calling predict()

At this point, we want to test how good our model is, so we need to use our trained model to make predictions on the test set.

At the end of training, the Trainer class loaded the best trained model in the pipeline. We simply use the predict function passing the test set to compute the predictions.

In [ ]:
predictions = trainer.predict(test_dataset_hf)
print('Evaluation metrics on the test set:')
# predict() return multiple information, such as predicted logits for each sample and
# evaluation metrics computed using the compute_metrics function.
# In this moment, we're interested in accessing the evaluation metric to evaluate how good our network is
predictions.metrics

Evaluation metrics on the test set:


{'test_loss': 0.25481417775154114,
 'test_precision': 0.6211849192100538,
 'test_recall': 0.3206672845227062,
 'test_f1': 0.4229828850855745,
 'test_accuracy': 0.9419434825360181,
 'test_runtime': 8.4194,
 'test_samples_per_second': 152.862,
 'test_steps_per_second': 2.494}

# Week 7 Submission Task:

In this lab, we conducted full finetuning, which involves finetuning all parameters. As these models scale in size, this operation becomes computationally expensive to carry out. Hence, researchers have started developing efficient ways to conduct finetuning. One of the first techniques in this direction is to select a specific number of layers to train, mainly focusing on the top layers (the last layers).

This top layer selection process is manually conducted and influenced by the task complexity and the pretrained model's knowledge quality. The workflow for this process is to freeze all model layers except the last layer (the classification layer) and monitor the performance. If you are not satisfied, you can try training more layers and so on. However, before deciding to finetune additional layers, it's advisable to adjust the hyperparameters, such as the number of learning steps, batch size, and learning rate. Adjusting these parameters is likely to enhance performance and is more cost-effective than expanding the number of trainable layers.

We want you to use this approach to finetune for the previous tasks, aiming to attain performance comparable to that of full finetuning.

What to submit:
- your notebook and any datafiles etc. that your code depends on
- NB: include (in the notebook) any pip install statements that your code relies on
- ensure that before submitting you test the code by restarting the runtime and then hitting run all
- a small report explaining what you did and what you observe in the results. The report can be included in your notebook or submitted as a separate document.

**Hint:**

There's no need to change the codebase; you only need to disable the gradient for the layers you don't want to train. In previous labs (Lab 6, Lab 5), we explained how to iterate through model parameters to count them. You may need to refer to that part as a starting point and think about how to use it here to disable the gradient.

### Note from TA
Don't need to update all the weights just the ones from the last layers or last n layers. Can use hugging face or the from scratch model the choice is yours.

# Resources

- Speech and Language Processing (3rd ed. draft) Dan Jurafsky and James H. Martin
  - [Chapter 11](https://web.stanford.edu/~jurafsky/slp3/11.pdf), Sections Sections 10.7-10.9, 11.1, 11.2, 11.4
- [Huggingface Tutorial for Token Classification](https://huggingface.co/docs/transformers/v4.17.0/en/tasks/token_classification)